# 02 -- Data Cleaning & Integration

**Project:** Early Disease Risk Predictor  
**Phase:** 2 -- Data Acquisition and Preprocessing  
**Notebook:** Cleaning and Integration  

This notebook covers:
1. Per-dataset cleaning: missing values, duplicates, outliers, type fixing
2. Schema harmonisation: aligning column names and value conventions
3. Dataset integration: merging into a single analysis-ready master table
4. Saving cleaned individual files and the merged dataset to `data/processed/`


In [5]:
import os
import numpy as np
import pandas as pd
from scipy import stats
import warnings

warnings.filterwarnings("ignore")
RAW_DIR  = os.path.join('..', 'data', 'raw')
PROC_DIR = os.path.join('..', 'data', 'processed')
os.makedirs(PROC_DIR, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
print('Directories ready.')


Directories ready.


## 1. Load Raw Datasets

In [6]:
pima   = pd.read_csv(os.path.join(RAW_DIR, 'pima_diabetes.csv'))
heart  = pd.read_csv(os.path.join(RAW_DIR, 'uci_heart_disease.csv'))
fram   = pd.read_csv(os.path.join(RAW_DIR, 'framingham.csv'))

print('PIMA   :', pima.shape)
print('Heart  :', heart.shape)
print('Framingham:', fram.shape)


PIMA   : (768, 9)
Heart  : (303, 14)
Framingham: (4240, 16)


## 2. Clean -- PIMA Indians Diabetes Dataset

Known issues:
- Columns `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI` contain biologically impossible zeros that represent missing values.
- No duplicate rows expected but will be checked.
- Outliers managed via IQR capping.


In [7]:
df = pima.copy()
print('--- PIMA: shape before cleaning ---', df.shape)

# 2.1 Replace impossible zeros with NaN 
zero_not_allowed = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in zero_not_allowed:
    n = (df[col] == 0).sum()
    df[col] = df[col].replace(0, np.nan)
    print(f'  {col}: {n} zeros replaced with NaN')

#  2.2 Impute missing values with median
print('\nMissing before imputation:')
print(df.isnull().sum()[df.isnull().sum() > 0])
for col in zero_not_allowed:
    df[col].fillna(df[col].median(), inplace=True)
print('Missing after imputation:', df.isnull().sum().sum())

# 2.3 Remove duplicates 
before = len(df)
df.drop_duplicates(inplace=True)
print(f'\nDuplicates removed: {before - len(df)}')

# 2.4 Outlier capping via IQR 
numeric_cols = df.select_dtypes(include=np.number).columns.drop('Outcome')
for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    capped = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower, upper)
    print(f'  {col}: {capped} values capped')

# 2.5 Add source tag
df['source'] = 'pima'
print('\n--- PIMA: shape after cleaning ---', df.shape)
pima_clean = df
pima_clean.head()


--- PIMA: shape before cleaning --- (768, 9)
  Glucose: 5 zeros replaced with NaN
  BloodPressure: 35 zeros replaced with NaN
  SkinThickness: 227 zeros replaced with NaN
  Insulin: 374 zeros replaced with NaN
  BMI: 11 zeros replaced with NaN

Missing before imputation:
Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64
Missing after imputation: 0

Duplicates removed: 0
  Pregnancies: 4 values capped
  Glucose: 0 values capped
  BloodPressure: 14 values capped
  SkinThickness: 87 values capped
  Insulin: 346 values capped
  BMI: 8 values capped
  DiabetesPedigreeFunction: 29 values capped
  Age: 9 values capped

--- PIMA: shape after cleaning --- (768, 10)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,source
0,6.000,148.000,72.000,35.000,125.000,33.600,0.627,50.000,1,pima
1,1.000,85.000,66.000,29.000,125.000,26.600,0.351,31.000,0,pima
2,8.000,183.000,64.000,29.000,125.000,23.300,0.672,32.000,1,pima
3,1.000,89.000,66.000,23.000,112.875,28.100,0.167,21.000,0,pima
4,0.000,137.000,40.000,35.000,135.875,43.100,1.200,33.000,1,pima


## 3. Clean -- UCI Heart Disease Dataset

Known issues:
- Columns `ca` and `thal` contain `NaN` (originally coded as `?` in source, already parsed during collection).
- `target` column has values > 1 (original severity 0-4); binarise to 0/1.
- Outliers managed via IQR capping.


In [8]:
df = heart.copy()
print('--- Heart: shape before cleaning ---', df.shape)

# 3.1 Show missing values
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

#  3.2 Impute ca and thal with mode 
for col in ['ca', 'thal']:
    mode_val = df[col].mode()[0]
    df[col].fillna(mode_val, inplace=True)
    print(f'  {col}: imputed with mode = {mode_val}')

#  3.3 Binarise target (0 = no disease, 1 = disease) 
df['target'] = (df['target'] > 0).astype(int)
print('\nTarget distribution after binarisation:')
print(df['target'].value_counts())

#  3.4 Fix dtypes 
for col in ['ca', 'thal']:
    df[col] = df[col].astype(float)

# 3.5 Remove duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f'\nDuplicates removed: {before - len(df)}')

#  3.6 Outlier capping via IQR 
numeric_cols = df.select_dtypes(include=np.number).columns.drop('target')
for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    capped = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower, upper)
    print(f'  {col}: {capped} values capped')

#  3.7 Add source tag 
df['source'] = 'uci_heart'
print('\n--- Heart: shape after cleaning ---', df.shape)
heart_clean = df
heart_clean.head()


--- Heart: shape before cleaning --- (303, 14)

Missing values:
ca      4
thal    2
dtype: int64
  ca: imputed with mode = 0.0
  thal: imputed with mode = 3.0

Target distribution after binarisation:
target
0    164
1    139
Name: count, dtype: int64

Duplicates removed: 0
  age: 0 values capped
  sex: 0 values capped
  cp: 23 values capped
  trestbps: 9 values capped
  chol: 5 values capped
  fbs: 45 values capped
  restecg: 0 values capped
  thalach: 1 values capped
  exang: 0 values capped
  oldpeak: 5 values capped
  slope: 0 values capped
  ca: 20 values capped
  thal: 0 values capped

--- Heart: shape after cleaning --- (303, 15)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target,source
0,63.000,1.000,1.500,145.000,233.000,0.000,2.000,150.000,0.000,2.300,3.000,0.000,6.000,0,uci_heart
1,67.000,1.000,4.000,160.000,286.000,0.000,2.000,108.000,1.000,1.500,2.000,2.500,3.000,1,uci_heart
2,67.000,1.000,4.000,120.000,229.000,0.000,2.000,129.000,1.000,2.600,2.000,2.000,7.000,1,uci_heart
3,37.000,1.000,3.000,130.000,250.000,0.000,0.000,187.000,0.000,3.500,3.000,0.000,3.000,0,uci_heart
4,41.000,0.000,2.000,130.000,204.000,0.000,2.000,172.000,0.000,1.400,1.000,0.000,3.000,0,uci_heart


## 4. Clean -- Framingham Heart Study Dataset

Known issues:
- Several columns have missing values: `education`, `cigsPerDay`, `BPMeds`, `totChol`, `BMI`, `heartRate`, `glucose`.
- Outliers managed via IQR capping.


In [9]:
df = fram.copy()
print('--- Framingham: shape before cleaning ---', df.shape)

#  4.1 Show missing values 
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

#  4.2 Impute: median for continuous, mode for categorical/binary 
median_cols = ['cigsPerDay', 'totChol', 'BMI', 'heartRate', 'glucose']
mode_cols   = ['education', 'BPMeds']

for col in median_cols:
    if col in df.columns:
        df[col].fillna(df[col].median(), inplace=True)
        print(f'  {col}: imputed with median')

for col in mode_cols:
    if col in df.columns:
        df[col].fillna(df[col].mode()[0], inplace=True)
        print(f'  {col}: imputed with mode')

print('\nMissing after imputation:', df.isnull().sum().sum())

#  4.3 Remove duplicates 
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Duplicates removed: {before - len(df)}')

#  4.4 Outlier capping via IQR 
numeric_cols = df.select_dtypes(include=np.number).columns.drop('TenYearCHD')
for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    capped = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower, upper)
    print(f'  {col}: {capped} values capped')

#  4.5 Add source tag 
df['source'] = 'framingham'
print('\n--- Framingham: shape after cleaning ---', df.shape)
fram_clean = df
fram_clean.head()


--- Framingham: shape before cleaning --- (4240, 16)

Missing values:
education     105
cigsPerDay     29
BPMeds         53
totChol        50
BMI            19
heartRate       1
glucose       388
dtype: int64
  cigsPerDay: imputed with median
  totChol: imputed with median
  BMI: imputed with median
  heartRate: imputed with median
  glucose: imputed with median
  education: imputed with mode
  BPMeds: imputed with mode

Missing after imputation: 0
Duplicates removed: 0
  male: 0 values capped
  age: 0 values capped
  education: 0 values capped
  currentSmoker: 0 values capped
  cigsPerDay: 12 values capped
  BPMeds: 124 values capped
  prevalentStroke: 25 values capped
  prevalentHyp: 0 values capped
  diabetes: 109 values capped
  totChol: 57 values capped
  sysBP: 126 values capped
  diaBP: 77 values capped
  BMI: 97 values capped
  heartRate: 76 values capped
  glucose: 262 values capped

--- Framingham: shape after cleaning --- (4240, 17)


,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD,source
0,1,39,4.000,0,0.000,0.000,0,0,0,195.000,106.000,70.000,26.970,80.000,77.000,0,framingham
1,0,46,2.000,0,0.000,0.000,0,0,0,250.000,121.000,81.000,28.730,95.000,76.000,0,framingham
2,1,48,1.000,1,20.000,0.000,0,0,0,245.000,127.500,80.000,25.340,75.000,70.000,0,framingham
3,0,61,3.000,1,30.000,0.000,0,1,0,225.000,150.000,95.000,28.580,65.000,103.000,1,framingham
4,0,46,3.000,1,23.000,0.000,0,0,0,285.000,130.000,84.000,23.100,85.000,85.000,0,framingham


## 5. Save Cleaned Individual Datasets

In [10]:
pima_clean.to_csv(os.path.join(PROC_DIR, 'pima_clean.csv'),  index=False)
heart_clean.to_csv(os.path.join(PROC_DIR, 'heart_clean.csv'), index=False)
fram_clean.to_csv(os.path.join(PROC_DIR, 'fram_clean.csv'),   index=False)

print('Saved:')
for name, df in [('pima_clean', pima_clean), ('heart_clean', heart_clean), ('fram_clean', fram_clean)]:
    print(f'  {name}.csv -- {df.shape}')


Saved:
  pima_clean.csv -- (768, 10)
  heart_clean.csv -- (303, 15)
  fram_clean.csv -- (4240, 17)


## 6. Schema Harmonisation

Before merging, we align the three datasets to a common schema:
- Rename columns to a unified naming convention
- Retain only the shared or mappable clinical features
- Add `disease_label` columns: `has_diabetes`, `has_heart_disease`, `has_hypertension`


In [11]:
#  PIMA: map to common schema 
pima_h = pima_clean[[
    'Age', 'BMI', 'Glucose', 'BloodPressure', 'Insulin', 'source'
]].rename(columns={
    'Age'           : 'age',
    'BMI'           : 'bmi',
    'Glucose'       : 'glucose',
    'BloodPressure' : 'blood_pressure',
    'Insulin'       : 'insulin',
})
pima_h['has_diabetes']      = pima_clean['Outcome']
pima_h['has_heart_disease'] = np.nan
pima_h['has_hypertension']  = np.nan

#  UCI Heart: map to common schema 
heart_h = heart_clean[[
    'age', 'chol', 'trestbps', 'source'
]].rename(columns={
    'chol'    : 'cholesterol',
    'trestbps': 'blood_pressure',
})
heart_h['glucose']           = np.nan
heart_h['bmi']               = np.nan
heart_h['insulin']           = np.nan
heart_h['has_diabetes']      = np.nan
heart_h['has_heart_disease'] = heart_clean['target']
heart_h['has_hypertension']  = np.nan

#  Framingham: map to common schema 
fram_h = fram_clean[[
    'age', 'BMI', 'glucose', 'sysBP', 'totChol', 'source'
]].rename(columns={
    'BMI'    : 'bmi',
    'sysBP'  : 'blood_pressure',
    'totChol': 'cholesterol',
})
fram_h['insulin']            = np.nan
fram_h['has_diabetes']       = np.nan
fram_h['has_heart_disease']  = fram_clean['TenYearCHD']
# Hypertension proxy: systolic BP >= 130 mmHg
fram_h['has_hypertension']   = (fram_clean['sysBP'] >= 130).astype(int)

print('Schema harmonised.')
print('PIMA harmonised     :', pima_h.shape)
print('Heart harmonised    :', heart_h.shape)
print('Framingham harmonised:', fram_h.shape)


Schema harmonised.
PIMA harmonised     : (768, 9)
Heart harmonised    : (303, 10)
Framingham harmonised: (4240, 10)


## 7. Dataset Integration

Concatenate the three harmonised DataFrames into a single master dataset.
Rows from each source retain their original label columns; columns not present in a given source are `NaN` and will be handled during the feature engineering phase.


In [13]:
master = pd.concat([pima_h, heart_h, fram_h], ignore_index=True)

print('Master dataset shape:', master.shape)
print('\nSource distribution:')
print(master['source'].value_counts())
print('\nMissing values per column:')
print(master.isnull().sum())
print('Missing values in master BEFORE post-integration imputation:')
print(master.isnull().sum())
print()

# ── Continuous features: median imputation ────────────────────────────────
continuous_cols = ['glucose', 'bmi', 'insulin', 'blood_pressure', 'cholesterol']
for col in continuous_cols:
    if col in master.columns:
        median_val = master[col].median()
        n_filled = master[col].isnull().sum()
        master[col].fillna(median_val, inplace=True)
        print(f'  {col}: {n_filled} NaNs filled with median = {median_val:.3f}')

# ── Label columns: fill unknown with 0 ───────────────────────────────────
label_cols = ['has_diabetes', 'has_heart_disease', 'has_hypertension']
for col in label_cols:
    n_filled = master[col].isnull().sum()
    master[col].fillna(0, inplace=True)
    master[col] = master[col].astype(int)
    print(f'  {col}: {n_filled} NaNs filled with 0')

print()
print('Missing values in master AFTER post-integration imputation:')
print(master.isnull().sum())
print()
print('Master shape:', master.shape)
master.head(10)

Master dataset shape: (5311, 10)

Source distribution:
source
framingham    4240
pima           768
uci_heart      303
Name: count, dtype: int64

Missing values per column:
age                     0
bmi                   303
glucose               303
blood_pressure          0
insulin              4543
source                  0
has_diabetes         4543
has_heart_disease     768
has_hypertension     1071
cholesterol           768
dtype: int64
Missing values in master BEFORE post-integration imputation:
age                     0
bmi                   303
glucose               303
blood_pressure          0
insulin              4543
source                  0
has_diabetes         4543
has_heart_disease     768
has_hypertension     1071
cholesterol           768
dtype: int64

  glucose: 303 NaNs filled with median = 79.000
  bmi: 303 NaNs filled with median = 25.910
  insulin: 4543 NaNs filled with median = 125.000
  blood_pressure: 0 NaNs filled with median = 125.000
  cholesterol: 768 NaNs

,age,bmi,glucose,blood_pressure,insulin,source,has_diabetes,has_heart_disease,has_hypertension,cholesterol
0,50.000,33.600,148.000,72.000,125.000,pima,1,0,0,234.000
1,31.000,26.600,85.000,66.000,125.000,pima,0,0,0,234.000
2,32.000,23.300,183.000,64.000,125.000,pima,1,0,0,234.000
3,21.000,28.100,89.000,66.000,112.875,pima,0,0,0,234.000
4,33.000,43.100,137.000,40.000,135.875,pima,1,0,0,234.000
5,30.000,25.600,116.000,74.000,125.000,pima,0,0,0,234.000
6,26.000,31.000,78.000,50.000,112.875,pima,1,0,0,234.000
7,29.000,35.300,115.000,72.000,125.000,pima,0,0,0,234.000
8,53.000,30.500,197.000,70.000,135.875,pima,1,0,0,234.000
9,54.000,32.300,125.000,96.000,125.000,pima,1,0,0,234.000


## 8. Save Master Dataset

In [14]:
MASTER_PATH = os.path.join(PROC_DIR, 'master.csv')
master.to_csv(MASTER_PATH, index=False)
print(f'Master dataset saved to: {MASTER_PATH}')
print(f'Final shape: {master.shape}')


Master dataset saved to: ..\data\processed\master.csv
Final shape: (5311, 10)


## 9. Cleaning Summary

In [15]:
summary = pd.DataFrame([
    {'Dataset': 'PIMA Diabetes',
     'Raw Shape': str(pima.shape),
     'Clean Shape': str(pima_clean.shape),
     'Missing Handled': 'Zeros -> NaN -> Median imputation',
     'Outlier Strategy': 'IQR 1.5x capping'},
    {'Dataset': 'UCI Heart Disease',
     'Raw Shape': str(heart.shape),
     'Clean Shape': str(heart_clean.shape),
     'Missing Handled': 'Mode imputation for ca, thal',
     'Outlier Strategy': 'IQR 1.5x capping'},
    {'Dataset': 'Framingham Heart Study',
     'Raw Shape': str(fram.shape),
     'Clean Shape': str(fram_clean.shape),
     'Missing Handled': 'Median for continuous, mode for categorical',
     'Outlier Strategy': 'IQR 1.5x capping'},
])
summary


,Dataset,Raw Shape,Clean Shape,Missing Handled,Outlier Strategy
0,PIMA Diabetes,"(768, 9)","(768, 10)",Zeros -> NaN -> Median imputation,IQR 1.5x capping
1,UCI Heart Disease,"(303, 14)","(303, 15)","Mode imputation for ca, thal",IQR 1.5x capping
2,Framingham Heart Study,"(4240, 16)","(4240, 17)","Median for continuous, mode for categorical",IQR 1.5x capping
